# EffortScope

## 1. Project Overview

Bu notebook, musteri destek ticket verilerini analiz ederek efor modelleme, kaynak analizi, musteri segmentasyonu ve tahminleme calismalari uretmek icin hazirlanmistir.

Calisma yaklasimi:
- Kaggle dataset'i kod ile otomatik indirilir.
- CSV dosyasi klasor icinde otomatik bulunur.
- Kolon isimleri farkli olsa bile benzer kolonlar eslestirilmeye calisilir.
- Gercek efor alani bulunmadigi icin sentetik ama gercekci bir efor modeli olusturulur.
- EDA, NLP, Pareto, kumeleme ve regresyon modelleme birlikte uygulanir.


## 2. How to Use

Bu notebook'u calistirmak icin hucreleri yukaridan asagiya sirayla calistirmaniz yeterlidir.

Notlar:
- Dataset manuel yuklenmez.
- KaggleHub ile otomatik indirme yapilir.
- Kolon isimleri beklenenden farkliysa fallback mekanizmalari devreye girer.
- Aciklamalar Turkce, kod bloklari English hazirlanmistir.


In [ ]:
%pip install -q kagglehub wordcloud seaborn scikit-learn


## 3. Data Loading

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import Markdown, display
except Exception:
    def Markdown(text):
        return text

    def display(*args, **kwargs):
        return None

from wordcloud import WordCloud
from collections import Counter

from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import kagglehub

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)


def find_best_csv(root_path: str | Path) -> Path:
    root = Path(root_path)
    csv_files = sorted(root.rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV file found under: {root}")

    def sort_key(file_path: Path):
        try:
            size = file_path.stat().st_size
        except OSError:
            size = 0
        return (-size, len(file_path.parts), file_path.name.lower())

    return sorted(csv_files, key=sort_key)[0]


dataset_path = kagglehub.dataset_download("tobiasbueck/multilingual-customer-support-tickets")
csv_path = find_best_csv(dataset_path)
df = pd.read_csv(csv_path)

print(f"Dataset folder: {dataset_path}")
print(f"Selected CSV: {csv_path.name}")
print(f"Shape: {df.shape}")
display(df.head())


## 4. Column Mapping Guide

Bu bolumde dataset kolonlari otomatik olarak yorumlanir. Amac, farkli isimlendirmelere ragmen benzer anlam tasiyan kolonlari tespit etmektir.

Ornek eslestirme mantigi:
- `description` benzeri kolonlar: `description`, `ticket_description`, `body`, `message`, `text`
- `priority` benzeri kolonlar: `priority`, `severity`, `urgency`, `importance`
- `customer` benzeri kolonlar: `customer`, `account`, `client`, `organization`

Eger dogrudan uygun kolon bulunamazsa notebook fallback mekanizmasi kullanir ve akisi durdurmaz.


In [ ]:
def normalize_name(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(value).strip().lower())


def find_column(columns, keywords, preferred_dtypes=None):
    normalized = {col: normalize_name(col) for col in columns}
    scored = []

    for col, norm in normalized.items():
        score = 0
        for keyword in keywords:
            k = normalize_name(keyword)
            if norm == k:
                score += 100
            elif k in norm:
                score += 20
        if score > 0:
            scored.append((score, col))

    if scored:
        scored.sort(reverse=True)
        return scored[0][1]

    if preferred_dtypes:
        dtype_candidates = [
            col for col in columns
            if any(dtype in str(df[col].dtype).lower() for dtype in preferred_dtypes)
        ]
        if dtype_candidates:
            return dtype_candidates[0]
    return None


column_map = {
    "description": find_column(
        df.columns,
        ["description", "ticket_description", "body", "message", "text", "issue", "problem"],
        preferred_dtypes=["object", "string"]
    ),
    "priority": find_column(
        df.columns,
        ["priority", "severity", "urgency", "importance", "impact"],
        preferred_dtypes=["object", "string"]
    ),
    "language": find_column(
        df.columns,
        ["language", "lang", "locale"],
        preferred_dtypes=["object", "string"]
    ),
    "product": find_column(
        df.columns,
        ["product", "service", "category", "topic", "issue_type"],
        preferred_dtypes=["object", "string"]
    ),
    "date": find_column(
        df.columns,
        ["created_at", "created", "date", "timestamp", "opened_at"]
    )
}

text_fallback = df.select_dtypes(include=["object", "string"]).fillna("").astype(str)
if column_map["description"] is None:
    if text_fallback.shape[1] > 0:
        df["fallback_description"] = text_fallback.apply(
            lambda row: " ".join([x for x in row if x.strip()]), axis=1
        )
    else:
        df["fallback_description"] = "support ticket"
    column_map["description"] = "fallback_description"

if column_map["priority"] is None:
    df["fallback_priority"] = pd.Series(
        rng.choice(["low", "medium", "high", "critical"], size=len(df), p=[0.30, 0.40, 0.22, 0.08])
    )
    column_map["priority"] = "fallback_priority"

mapping_df = pd.DataFrame({
    "logical_field": list(column_map.keys()),
    "mapped_column": list(column_map.values())
})

display(mapping_df)


## 5. Data Cleaning

In [ ]:
df = df.copy()
df.columns = [str(col).strip() for col in df.columns]
df = df.drop_duplicates().reset_index(drop=True)

description_col = column_map["description"]
priority_col = column_map["priority"]
language_col = column_map["language"]
product_col = column_map["product"]
date_col = column_map["date"]

df[description_col] = df[description_col].fillna("support ticket").astype(str).str.strip()
df.loc[df[description_col].eq(""), description_col] = "support ticket"
df[priority_col] = df[priority_col].fillna("medium").astype(str).str.strip().str.lower()

priority_normalization = {
    "low": "low",
    "medium": "medium",
    "med": "medium",
    "normal": "medium",
    "high": "high",
    "urgent": "high",
    "critical": "critical",
    "sev1": "critical",
    "sev2": "high",
    "sev3": "medium",
    "sev4": "low"
}

df[priority_col] = df[priority_col].map(lambda x: priority_normalization.get(x, x))
valid_priorities = {"low", "medium", "high", "critical"}
df.loc[~df[priority_col].isin(valid_priorities), priority_col] = "medium"

if language_col is not None:
    df[language_col] = df[language_col].fillna("unknown").astype(str).str.strip()

if product_col is not None:
    df[product_col] = df[product_col].fillna("unknown").astype(str).str.strip()

if date_col is not None:
    parsed_dates = pd.to_datetime(df[date_col], errors="coerce")
    if parsed_dates.notna().sum() > 0:
        df["created_at_clean"] = parsed_dates
    else:
        date_col = None

print(f"Rows after cleaning: {len(df):,}")
display(df.head())


## 6. Missing Values

In [ ]:
missing_summary = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
missing_summary.columns = ["column", "missing_count"]
missing_summary["missing_ratio"] = missing_summary["missing_count"] / len(df)

display(missing_summary.head(15))

plt.figure(figsize=(12, 5))
sns.barplot(data=missing_summary.head(15), x="missing_count", y="column", palette="crest")
plt.title("En Fazla Eksik Degere Sahip Kolonlar")
plt.xlabel("Eksik Deger Sayisi")
plt.ylabel("Kolon")
plt.tight_layout()
plt.show()


## 7. Outlier Handling

Bu calismada aykiri degerler silinmemistir. Bunun yerine IQR yontemi ile alt ve ust sinirlar hesaplanmis, ilgili sayisal degiskenler bu sinirlara kirpilmisir.


## 8. Feature Engineering

Bu dataset'te gercek efor olmadigi icin sentetik ama gercekci bir model olusturulmustur.

Uretilen degiskenler:
- `description_length`
- `word_count`
- `complexity_score`
- `created_day_of_week`
- `is_weekend`
- `customer_name`
- `effort_hours`
- `resolution_time`

Efor formulu:

`effort_hours = (word_count * 0.04) + (priority_weight * 0.8) + random_noise`

Burada `random_noise` 0.5 ile 2.0 arasinda uretilmistir ve sabit random seed ile tekrar uretilebilir yapi korunmustur.


In [ ]:
priority_weight_map = {"low": 1, "medium": 2, "high": 3, "critical": 5}
company_names = [
    "NovaTech", "BlueOrbit", "Anka Retail", "DataBridge", "OrionTel",
    "LunaPay", "PeakLogistics", "CloudHarbor", "ZenithCare", "AtlasWorks"
]

description_text = df[description_col].fillna("support ticket").astype(str)
df["description_length"] = description_text.str.len()
df["word_count"] = description_text.str.split().map(len)

priority_weight = df[priority_col].map(priority_weight_map).fillna(2)
df["complexity_score"] = (
    (df["description_length"] / max(df["description_length"].median(), 1)) * 0.4 +
    (df["word_count"] / max(df["word_count"].median(), 1)) * 0.4 +
    (priority_weight / 5) * 0.2
).round(3)

if "created_at_clean" in df.columns:
    df["created_day_of_week"] = df["created_at_clean"].dt.day_name()
else:
    simulated_dates = pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, len(df)), unit="D")
    df["created_at_clean"] = simulated_dates
    df["created_day_of_week"] = simulated_dates.day_name()

df["is_weekend"] = df["created_day_of_week"].isin(["Saturday", "Sunday"]).astype(int)
df["customer_name"] = rng.choice(company_names, size=len(df))

noise = rng.uniform(0.5, 2.0, len(df))
df["effort_hours"] = (df["word_count"] * 0.04) + (priority_weight * 0.8) + noise
df["effort_hours"] = df["effort_hours"].round(2)

resolution_factor = rng.uniform(1.1, 2.4, len(df))
df["resolution_time"] = (df["effort_hours"] * resolution_factor).round(2)

numeric_for_capping = ["description_length", "word_count", "complexity_score", "effort_hours", "resolution_time"]

def iqr_cap(series: pd.Series) -> pd.Series:
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return series.clip(lower, upper)

for col in numeric_for_capping:
    df[col] = iqr_cap(df[col])

display(df[[description_col, priority_col, "description_length", "word_count", "complexity_score", "effort_hours", "resolution_time"]].head())


## 9. NLP Analysis

In [ ]:
stopwords = {
    "the", "and", "for", "with", "that", "this", "from", "have", "been", "your", "you", "are",
    "bir", "ve", "ile", "icin", "cok", "ama", "gibi", "daha", "kadar", "ticket", "support"
}

def tokenize_text(series: pd.Series) -> list[str]:
    text = " ".join(series.fillna("").astype(str).tolist()).lower()
    tokens = re.findall(r"[a-zA-Z]{3,}", text)
    return [token for token in tokens if token not in stopwords]

if description_col in df.columns and df[description_col].fillna("").astype(str).str.len().sum() > 0:
    tokens = tokenize_text(df[description_col])
    top_words = pd.DataFrame(Counter(tokens).most_common(15), columns=["word", "frequency"])
    display(top_words)

    if not top_words.empty:
        plt.figure(figsize=(12, 5))
        sns.barplot(data=top_words, x="frequency", y="word", palette="mako")
        plt.title("En Sik Gecen Kelimeler")
        plt.xlabel("Frekans")
        plt.ylabel("Kelime")
        plt.tight_layout()
        plt.show()

    if tokens:
        wordcloud = WordCloud(width=1200, height=500, background_color="white", colormap="viridis").generate(" ".join(tokens))
        plt.figure(figsize=(14, 6))
        plt.imshow(wordcloud, interpolation="bilinear")
        plt.axis("off")
        plt.title("WordCloud")
        plt.show()
    else:
        print("Description metninden anlamli token uretilemedi.")
else:
    print("Description benzeri kolon bulunamadigi icin NLP analizi guvenli bicimde atlandi.")


## 10. EDA

In [ ]:
display(df.describe(include="all").transpose().head(20))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(df["effort_hours"], bins=30, kde=True, ax=axes[0], color="#1f77b4")
axes[0].set_title("Effort Hours Distribution")

priority_order = ["low", "medium", "high", "critical"]
priority_counts = df[priority_col].value_counts().reindex(priority_order).fillna(0)
sns.barplot(x=priority_counts.index, y=priority_counts.values, ax=axes[1], palette="rocket")
axes[1].set_title("Priority Distribution")
axes[1].set_xlabel("Priority")
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.show()


## 11. Pareto Analysis

In [ ]:
pareto_col = product_col if product_col is not None else priority_col
pareto_df = df[pareto_col].fillna("unknown").astype(str).value_counts().reset_index()
pareto_df.columns = [pareto_col, "ticket_count"]
pareto_df["cum_pct"] = pareto_df["ticket_count"].cumsum() / pareto_df["ticket_count"].sum() * 100

fig, ax1 = plt.subplots(figsize=(12, 6))
sns.barplot(data=pareto_df, x=pareto_col, y="ticket_count", ax=ax1, color="#4c78a8")
ax1.set_title(f"Pareto Analysis - {pareto_col}")
ax1.set_xlabel(pareto_col)
ax1.set_ylabel("Ticket Count")
ax1.tick_params(axis="x", rotation=45)

ax2 = ax1.twinx()
ax2.plot(pareto_df.index, pareto_df["cum_pct"], color="#e45756", marker="o")
ax2.axhline(80, color="gray", linestyle="--")
ax2.set_ylabel("Cumulative %")
ax2.set_ylim(0, 110)
plt.tight_layout()
plt.show()

display(pareto_df.head(10))


## 12. Heatmap

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numeric_cols].corr(numeric_only=True)

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=False)
plt.title("Numeric Feature Correlation Heatmap")
plt.tight_layout()
plt.show()


## 13. Boxplot

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x=priority_col, y="effort_hours", order=["low", "medium", "high", "critical"], palette="Set2")
plt.title("Priority Bazinda Effort Hours Boxplot")
plt.xlabel("Priority")
plt.ylabel("Effort Hours")
plt.tight_layout()
plt.show()


## 14. Clustering

Bu bolumde musteri bazli segmentasyon icin `KMeans` kullanilmistir. Kumeleme sayisi sabit olarak `4` secilmistir.


In [ ]:
customer_agg = (
    df.groupby("customer_name")
    .agg(
        avg_effort_hours=("effort_hours", "mean"),
        avg_resolution_time=("resolution_time", "mean"),
        avg_complexity_score=("complexity_score", "mean"),
        ticket_count=("customer_name", "size"),
        weekend_ratio=("is_weekend", "mean")
    )
    .reset_index()
)

cluster_features = ["avg_effort_hours", "avg_resolution_time", "avg_complexity_score", "ticket_count", "weekend_ratio"]
kmeans = KMeans(n_clusters=4, random_state=RANDOM_SEED, n_init=20)
customer_agg["cluster"] = kmeans.fit_predict(customer_agg[cluster_features])

display(customer_agg.sort_values(["cluster", "avg_effort_hours"]))

plt.figure(figsize=(12, 6))
sns.scatterplot(
    data=customer_agg,
    x="avg_effort_hours",
    y="ticket_count",
    hue="cluster",
    size="avg_resolution_time",
    sizes=(100, 800),
    palette="tab10"
)
plt.title("Customer Segmentation with KMeans")
plt.xlabel("Average Effort Hours")
plt.ylabel("Ticket Count")
plt.tight_layout()
plt.show()


## 15. Regression Modeling

Tahminleme hedefi olarak `effort_hours` kullanilmistir. Veri `%80 / %20` oraninda egitim ve test olarak ayrilmistir. Kategorik alanlarda One-Hot Encoding uygulanmistir.


In [ ]:
def compute_rmse(y_true, y_pred):
    import numpy as np
    try:
        from sklearn.metrics import root_mean_squared_error
        return float(root_mean_squared_error(y_true, y_pred))
    except Exception:
        from sklearn.metrics import mean_squared_error
        return float(np.sqrt(mean_squared_error(y_true, y_pred)))
feature_candidates = [
    "description_length", "word_count", "complexity_score", "is_weekend", "resolution_time",
    priority_col, "created_day_of_week", "customer_name"
]

if language_col is not None:
    feature_candidates.append(language_col)
if product_col is not None:
    feature_candidates.append(product_col)

feature_cols = [col for col in feature_candidates if col in df.columns]
model_df = df[feature_cols + ["effort_hours"]].copy()

X = model_df.drop(columns=["effort_hours"])
y = model_df["effort_hours"]

categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_features = [col for col in X.columns if col not in categorical_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED
)

numeric_preprocess_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numeric_preprocess_plain = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_scaled = ColumnTransformer([
    ("num", numeric_preprocess_scaled, numeric_features),
    ("cat", categorical_preprocess, categorical_features)
])

preprocessor_plain = ColumnTransformer([
    ("num", numeric_preprocess_plain, numeric_features),
    ("cat", categorical_preprocess, categorical_features)
])

models = {
    "Linear Regression": Pipeline([
        ("preprocessor", preprocessor_scaled),
        ("model", LinearRegression())
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor_plain),
        ("model", RandomForestRegressor(n_estimators=250, random_state=RANDOM_SEED))
    ]),
    "Gradient Boosting": Pipeline([
        ("preprocessor", preprocessor_plain),
        ("model", GradientBoostingRegressor(random_state=RANDOM_SEED))
    ])
}

results = []
fitted_models = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    results.append({
        "model": name,
        "MAE": mean_absolute_error(y_test, preds),
        "RMSE": compute_rmse(y_test, preds),
        "R2": r2_score(y_test, preds)
    })
    fitted_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values("RMSE")
display(results_df)


## 16. Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metric_list = ["MAE", "RMSE", "R2"]
palettes = ["Blues_r", "Greens_r", "Oranges"]

for ax, metric, palette in zip(axes, metric_list, palettes):
    sns.barplot(data=results_df, x="model", y=metric, palette=palette, ax=ax)
    ax.set_title(f"Model Comparison - {metric}")
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


## 17. Feature Importance

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_pipeline = fitted_models[best_model_name]

if best_model_name in {"Random Forest", "Gradient Boosting"}:
    preprocessor = best_pipeline.named_steps["preprocessor"]
    model = best_pipeline.named_steps["model"]
    feature_names = preprocessor.get_feature_names_out()
    importances = model.feature_importances_
    feature_importance_df = (
        pd.DataFrame({"feature": feature_names, "importance": importances})
        .sort_values("importance", ascending=False)
        .head(15)
    )
    display(feature_importance_df)

    plt.figure(figsize=(12, 6))
    sns.barplot(data=feature_importance_df, x="importance", y="feature", palette="viridis")
    plt.title(f"Feature Importance - {best_model_name}")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()
else:
    print("En iyi model Linear Regression oldugu icin standart feature importance grafigi yerine katsayi bazli yorum tercih edilmelidir.")


## 18. Insights

Bu bolumde analiz ciktlari is ve operasyon perspektifinden yorumlanir. `effort_hours` sentetik olarak uretildigi icin burada paylasilan bulgular gercek operasyonun birebir performans vaadi olarak degil, analitik simulasyon ve karar destek bakisiyla okunmalidir.


In [ ]:
best_model_row = results_df.sort_values("RMSE").iloc[0]
best_model_name = best_model_row["model"]
complexity_effort_corr = df[["complexity_score", "effort_hours"]].corr().iloc[0, 1]
priority_effort = df.groupby(priority_col)["effort_hours"].mean().sort_values(ascending=False)
busiest_day = df["created_day_of_week"].value_counts().idxmax()
busiest_day_count = int(df["created_day_of_week"].value_counts().max())
weekend_effort = df.groupby("is_weekend")["effort_hours"].mean().to_dict()
weekend_label = "hafta sonu" if weekend_effort.get(1, 0) > weekend_effort.get(0, 0) else "hafta ici"
cluster_effort = customer_agg.groupby("cluster")["avg_effort_hours"].mean().sort_values(ascending=False)
most_expensive_cluster = int(cluster_effort.index[0])
most_expensive_cluster_effort = float(cluster_effort.iloc[0])

insight_md = f"""
## Veri Tabanli Bulgular

- **En iyi model:** {best_model_name}. Test setinde MAE={best_model_row['MAE']:.3f}, RMSE={best_model_row['RMSE']:.3f}, R2={best_model_row['R2']:.3f} degerleri elde edilmistir.
- **MAE / RMSE / R2 yorumu:** MAE ortalama mutlak sapmayi, RMSE buyuk hatalara daha duyarli genel hata seviyesini, R2 ise modelin sentetik hedef varyansini ne kadar acikladigini gosterir.
- **Model guvenilirligi:** Model teknik olarak guclu gorunmektedir; ancak hedef degisken sentetik oldugu icin bu performans gercek operasyonel SLA basarisi degil, analitik mantigin ne kadar iyi ogrenildigini ifade eder.
- **Complexity ve effort iliskisi:** `complexity_score` ile `effort_hours` arasindaki korelasyon **{complexity_effort_corr:.2f}** seviyesindedir. Bu da daha karmasik ticket'larin daha yuksek efor tukettigini destekler.
- **Musteri segmentleri:** Ortalama eforu en yuksek segment **Cluster {most_expensive_cluster}** olup ortalama **{most_expensive_cluster_effort:.2f} saat** seviyesindedir.
- **Yogunluk gunleri:** En yuksek ticket hacmi **{busiest_day}** gununde gorulmektedir ({busiest_day_count:,} ticket). Ortalama efor tarafinda ise **{weekend_label}** yukunun cok az farkla daha baskin oldugu izlenmektedir.
- **Maliyetli priority:** Ortalama efor bazinda en maliyetli seviye **{priority_effort.index[0]}** olup ortalama **{priority_effort.iloc[0]:.2f} saat** efor gerektirmektedir.

**Teknik not:** RMSE compatibility helper was used to support different scikit-learn versions.
"""
display(Markdown(insight_md))


## 19. Recommendations

1. **Complexity bazli uzman yonlendirme**
- Sorun: Yuksek complexity ticket'lar standart akista daha fazla zaman ve tekrar isleme neden olabilir.
- Veriyle gozlem: `complexity_score` ile `effort_hours` arasinda guclu pozitif iliski vardir.
- Onerilen aksiyon: Complexity esigi uzerindeki ticket'lar ilk asamada uzman ekip veya ikinci seviye destek kuyruguna aktarilmalidir.
- Beklenen operasyonel etki: Daha hizli cozum, daha az tekrar islem ve daha dengeli ekip kullanimi.

2. **Gun bazli kapasite planlama**
- Sorun: Ticket hacmi haftaya esit dagilmiyor.
- Veriyle gozlem: En yuksek hacim `Wednesday` gununde goruluyor; yogunluk hafta ici ve hafta sonu arasinda tamamen kaybolmuyor.
- Onerilen aksiyon: Vardiya ve back-up kapasite planlari gun bazli talep desenine gore optimize edilmelidir.
- Beklenen operasyonel etki: Backlog birikimi ve SLA sapmasi riski azalir.

3. **Yuksek efor tuketen segmentler icin SLA ve otomasyon tasarimi**
- Sorun: Bazi musteri segmentleri sistematik olarak daha yuksek efor tuketiyor.
- Veriyle gozlem: KMeans segmentasyonunda belirli cluster'lar daha yuksek ortalama efor uretiyor.
- Onerilen aksiyon: Bu segmentler icin ozel SLA, self-service icerikleri ve tekrar eden konu bazli otomasyon senaryolari tasarlanmalidir.
- Beklenen operasyonel etki: Kaynak tuketimi daha ongorulebilir hale gelir ve destek maliyeti kontrol altina alinabilir.


## 20. Executive Summary

- Proje, customer support ticket verilerinden efor ve kaynak kullanimini anlamak icin gelistirildi.
- Veri kaynagi olarak `tobiasbueck/multilingual-customer-support-tickets` dataseti kullanildi ve yerel proje klasorune kaydedildi.
- Gercek efor alani olmadigi icin `effort_hours` ve `resolution_time` analitik gosterim amaciyla sentetik olarak uretildi.
- Complexity, priority ve metin yogunlugu arttikca sentetik eforun da anlamli bicimde arttigi goruldu.
- Musteri segmentasyonu, bazi musteri gruplarinin daha yuksek ortalama efor tukettigini gosterdi.
- Regresyon modellemede en iyi sonuc **Gradient Boosting** ile elde edildi.
- Bu performans gercek operasyonel tahmin basarisi olarak degil, sentetik efor mantiginin basarili sekilde ogrenilmesi olarak yorumlanmalidir.
- Onerilen aksiyon alanlari: uzman yonlendirme, gun bazli kapasite planlama ve segment bazli SLA / otomasyon tasarimi.
